# Contrail Labeling Helper

Interactive tool for labeling flight contrails.

## Labeling Protocol
- **Blocked**: Flight path obscured (clouds, etc.)
- **Clear**: No contrail visible
- **Dissipate < 10**: Contrail dissipates in < 10 seconds
- **Dissipate > 10**: Contrail dissipates in > 10 seconds
- **Persistent**: Contrail persists for extended time

In [9]:
import pandas as pd
import numpy as np
import cv2
import os
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import json

import utils.adsb_utils as adsb_utils
import utils.projection_utils as proj_utils
from utils.image_data_utils import get_image_data_uwisc
# reload import


In [2]:
import importlib
importlib.reload(adsb_utils)
importlib.reload(proj_utils)

<module 'utils.projection_utils' from '/Users/shrenikborad/pless/contrails/utils/projection_utils.py'>

## Configuration
Set your date, camera side, and paths below:

In [4]:
# Configuration
DATE_STR = "2025-03-13"  # Change this to your date
CAMERA_SIDE = "east"     # 'east' or 'south'

# Paths
ADSB_CSV_PATH = f"/Users/shrenikborad/pless/easy_adsb/wisconsin_2025_03_13.csv"
CAMERA_PARAMS_PATH = f"/Users/shrenikborad/pless/contrails/uwisc/east/camera_params.json"
BASE_DIR = f'/Users/shrenikborad/pless/contrails/downloaded_images/east/2025-03-13'
CAMERA_NAME = f"wisconsin_{CAMERA_SIDE}"

# Output path for labels
LABELS_OUTPUT_PATH = f"./contrail_labels_{DATE_STR}_{CAMERA_NAME}.csv"

print(f"Date: {DATE_STR}")
print(f"Camera: {CAMERA_NAME}")
print(f"Labels will be saved to: {LABELS_OUTPUT_PATH}")

Date: 2025-03-13
Camera: wisconsin_east
Labels will be saved to: ./contrail_labels_2025-03-13_wisconsin_east.csv


## Load Data

In [5]:
print("Projecting to image coordinates...")
intrinsics, distortion, rvec, tvec, origin_gps = proj_utils.load_camera_parameters(CAMERA_PARAMS_PATH)


Projecting to image coordinates...


In [6]:
# Load ADSB data
print("Loading ADSB data...")
df = adsb_utils.read_adsblol_csv(ADSB_CSV_PATH, origin_gps=origin_gps )

# Filter to daytime hours
from_dt = pd.to_datetime(f"{DATE_STR} 06:00:00").tz_localize('America/Phoenix').tz_convert('UTC')
to_dt = pd.to_datetime(f"{DATE_STR} 17:00:00").tz_localize('America/Phoenix').tz_convert('UTC')
df['time'] = pd.to_datetime(df['time'])
df = df[(df['time'] >= from_dt) & (df['time'] < to_dt)]

print(f"Loaded {len(df)} ADSB pings")

# Upsample flight data
print("Upsampling flight data...")
df_upsampled = adsb_utils.get_upsampled_df_for_day(df, max_range_m=100000)



Loading ADSB data...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:167: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['flight'] = df.groupby(['icao', 'registration'])['flight'].transform(lambda s: s.ffill().bfill())


Loaded 228614 ADSB pings
Upsampling flight data...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time'] = pd.to_datetime(df['time'])
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:111: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['alt_gnss_meters'] = df['alt_gnss_meters'].astype(float)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Upsampling all aircraft...
Processing 955 unique aircraft...

Processed 10 aircraft...
Processed 20 aircraft...
Processed 30 aircraft...
Processed 40 aircraft...
Processed 50 aircraft...
Processed 60 aircraft...
Processed 70 aircraft...
Processed 80 aircraft...
Processed 90 aircraft...
Processed 100 aircraft...
Processed 110 aircraft...
Processed 120 aircraft...
Processed 130 aircraft...
Processed 140 aircraft...
Processed 150 aircraft...
Processed 160 aircraft...
Processed 170 aircraft...
Processed 180 aircraft...
Processed 190 aircraft...
Processed 200 aircraft...
Processed 210 aircraft...
Processed 220 aircraft...
Processed 230 aircraft...
Processed 240 aircraft...
Processed 250 aircraft...
Processed 260 aircraft...
Processed 270 aircraft...
Processed 280 aircraft...
Processed 290 aircraft...
Processed 300 aircraft...
Processed 310 aircraft...
Processed 320 aircraft...
Processed 330 aircraft...
Processed 340 aircraft...
Processed 350 aircraft...
Processed 360 aircraft...
Processed 3

/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:139: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_upsampled[['lat', 'lon', 'alt_gnss_meters']].applymap(lambda x: isinstance(x, str)).any(axis=1)


In [10]:
# Load camera parameters and project to image coordinates

image_x, image_y, cam_distance = proj_utils.gps_to_camxy_vasha_fixed(
    df_upsampled['lat'].values,
    df_upsampled['lon'].values,
    df_upsampled['alt_gnss_meters'].values,
    cam_k=intrinsics,
    cam_r=rvec,
    cam_t=tvec,
    camera_gps=origin_gps,
    distortion=distortion
)

df_upsampled['image_x'] = image_x
df_upsampled['image_y'] = image_y
df_upsampled['cam_distance'] = cam_distance

# Load image metadata
print("Loading image metadata...")
image_df = get_image_data_uwisc(BASE_DIR, "2025-03-13")
image_df = image_df[(image_df['time'] >= from_dt) & (image_df['time'] < to_dt)]
max_time = image_df['time'].max() + timedelta(seconds=1)
min_time = image_df['time'].min() - timedelta(seconds=1)
image_df = image_df.sort_values('time').reset_index(drop=True)
df_upsampled = df_upsampled[
    (df_upsampled['time'] >= min_time) & 
    (df_upsampled['time'] <= max_time)
].copy()

print(f"Loaded {len(image_df)} images")
print(f"Time range: {image_df['time'].min()} to {image_df['time'].max()}")

Loading image metadata...
Total images found: 8640
Total images between 3 pm and 4 pm utc: 3960
                       time            image_file
0 2025-03-13 13:00:08+00:00  08_00_08.trig+00.jpg
1 2025-03-13 13:00:18+00:00  08_00_18.trig+00.jpg
2 2025-03-13 13:00:28+00:00  08_00_28.trig+00.jpg
3 2025-03-13 13:00:38+00:00  08_00_38.trig+00.jpg
4 2025-03-13 13:00:48+00:00  08_00_48.trig+00.jpg
Loaded 3960 images
Time range: 2025-03-13 13:00:08+00:00 to 2025-03-13 23:59:58+00:00


In [11]:
# filter
# 1477.53,609.57,-110.8435989781433,32.38439428076198,2357.407900804103,mountain_right_peak
df_upsampled = df_upsampled[
    ~ ((df_upsampled['image_y'] > 609.57) &
    (df_upsampled['distance_m'] > adsb_utils.haversine_km(32.38439428076198, -110.8435989781433, origin_gps[0], origin_gps[1]) * 1000)
)]

In [12]:
# Get unique flights visible in the time window
# A flight is visible if it has valid image coordinates
image = image_df.iloc[0]
cv2_image = cv2.imread(BASE_DIR + "/"+ image['image_file'])
image_height, image_width = cv2_image.shape[:2]
df_visible = df_upsampled[
    (df_upsampled['image_x'].notna()) & 
    (df_upsampled['image_y'].notna()) &
    (df_upsampled['image_x'] >= 0) &
    (df_upsampled['image_y'] >= 0) &
    (df_upsampled['image_x'] < image_width) &
    (df_upsampled['image_y'] < image_height)
].copy()

# Get flight summary
flight_summary = df_visible.groupby('ident').agg({
    'time': ['min', 'max', 'count'],
    'alt_gnss_meters': ['min', 'max', 'mean'],
    'image_x': 'mean',
    'image_y': 'mean'
}).reset_index()

flight_summary.columns = ['ident', 'time_appear', 'time_disappear', 'n_points', 
                          'alt_min', 'alt_max', 'alt_mean', 'avg_x', 'avg_y']

# Convert altitude to feet for display
flight_summary['alt_min_ft'] = (flight_summary['alt_min'] * 3.28084).round(0)
flight_summary['alt_max_ft'] = (flight_summary['alt_max'] * 3.28084).round(0)

print(f"Found {len(flight_summary)} unique flights visible in frame")
flight_summary.head(10)

Found 598 unique flights visible in frame


,ident,time_appear,time_disappear,n_points,alt_min,alt_max,alt_mean,avg_x,avg_y,alt_min_ft,alt_max_ft
0,AAL118,2025-03-13 16:19:27+00:00,2025-03-13 16:21:12+00:00,106,10629.900000,10637.520,10632.955189,2335.798027,1029.546022,34875.0,34900.0
1,AAL1473,2025-03-13 21:50:14+00:00,2025-03-13 21:50:41+00:00,28,3300.984000,3360.420,3339.846000,2486.293318,1145.927813,10830.0,11025.0
2,AAL1578,2025-03-13 18:46:12+00:00,2025-03-13 18:50:48+00:00,277,10614.660000,10629.900,10619.817942,933.797828,791.987235,34825.0,34875.0
3,AAL1598,2025-03-13 17:58:14+00:00,2025-03-13 17:58:42+00:00,29,3305.175000,3375.660,3344.785862,2487.701492,1145.744360,10844.0,11075.0
4,AAL16,2025-03-13 20:55:40+00:00,2025-03-13 20:57:26+00:00,107,10652.760000,10660.380,10655.572991,2341.268459,1028.351536,34950.0,34975.0
5,AAL1647,2025-03-13 21:01:27+00:00,2025-03-13 21:01:55+00:00,29,3268.133333,3383.280,3323.692184,2486.965079,1146.122411,10722.0,11100.0
6,AAL1757,2025-03-13 22:18:55+00:00,2025-03-13 22:19:57+00:00,63,2446.020000,3139.440,2809.300476,1872.386447,1188.180339,8025.0,10300.0
7,AAL1835,2025-03-13 18:22:08+00:00,2025-03-13 18:25:50+00:00,223,2491.740000,3680.460,3036.279552,2131.902200,1136.488542,8175.0,12075.0
8,AAL1902,2025-03-13 20:48:40+00:00,2025-03-13 20:50:34+00:00,115,11250.168000,11269.980,11261.902800,2315.394506,1016.795972,36910.0,36975.0
9,AAL1979,2025-03-13 23:58:05+00:00,2025-03-13 23:58:31+00:00,27,3352.800000,3385.185,3363.101111,2487.720712,1145.418303,11000.0,11106.0


## Initialize Labels DataFrame

In [13]:
# Initialize or load existing labels
LABEL_OPTIONS = ['', 'Blocked', 'Clear', 'Dissipate < 10', 'Dissipate > 10', 'Persistent']

labels_df = flight_summary[['ident', 'time_appear', 'time_disappear', 
                                'alt_min', 'alt_max', 'alt_min_ft', 'alt_max_ft']].copy()
labels_df['alt_appear'] = labels_df['alt_min']
labels_df['alt_disappear'] = labels_df['alt_max']
labels_df['label'] = ''
labels_df['notes'] = ''
labels_df['section'] = 1  # For tracking splits

print(f"Labels dataframe has {len(labels_df)} entries")
labels_df.head()

Labels dataframe has 598 entries


,ident,time_appear,time_disappear,alt_min,alt_max,alt_min_ft,alt_max_ft,alt_appear,alt_disappear,label,notes,section
0,AAL118,2025-03-13 16:19:27+00:00,2025-03-13 16:21:12+00:00,10629.900,10637.52,34875.0,34900.0,10629.900,10637.52,,,1
1,AAL1473,2025-03-13 21:50:14+00:00,2025-03-13 21:50:41+00:00,3300.984,3360.42,10830.0,11025.0,3300.984,3360.42,,,1
2,AAL1578,2025-03-13 18:46:12+00:00,2025-03-13 18:50:48+00:00,10614.660,10629.90,34825.0,34875.0,10614.660,10629.90,,,1
3,AAL1598,2025-03-13 17:58:14+00:00,2025-03-13 17:58:42+00:00,3305.175,3375.66,10844.0,11075.0,3305.175,3375.66,,,1
4,AAL16,2025-03-13 20:55:40+00:00,2025-03-13 20:57:26+00:00,10652.760,10660.38,34950.0,34975.0,10652.760,10660.38,,,1


## Interactive Labeling Interface

In [14]:
# Export data for HTML labeler
import json

def export_labeling_data(image_df, df_upsampled, labels_df, base_dir, output_json_path):
    """Export all data needed for the HTML labeling interface."""
    
    frames = []
    for idx, row in image_df.iterrows():
        t = row['time']
        
        # Get flights at this time
        flights_at_time = df_upsampled[df_upsampled['time'] == t].copy()
        flights_at_time = flights_at_time[
            (flights_at_time['image_x'].notna()) & 
            (flights_at_time['image_y'].notna()) &
            (flights_at_time['image_x'] >= 0) &
            (flights_at_time['image_y'] >= 0) &
            (flights_at_time['image_x'] < 3000) &  # Filter unrealistic values
            (flights_at_time['image_y'] < 3000)
        ]
        
        flights_list = []
        for _, f in flights_at_time.iterrows():
            flights_list.append({
                'ident': f['ident'],
                'x': float(f['image_x']),
                'y': float(f['image_y']),
                'alt_ft': round(f['alt_gnss_meters'] * 3.28084),
                'heading': float(f['heading']) if 'heading' in f and pd.notna(f['heading']) else 0
            })
        
        frames.append({
            'idx': idx,
            'image_file': row['image_file'],
            'time_utc': t.strftime('%Y-%m-%d %H:%M:%S'),
            'time_local': t.tz_convert('America/Chicago').strftime('%H:%M:%S'),
            'flights': flights_list
        })
    
    # Flight summary for labels
    flights_in_frames = set()
    for frame in frames:
        for flight in frame['flights']:
            flights_in_frames.add(flight['ident'])
    flight_list = []
    for _, row in labels_df.iterrows():
        if row['ident'] not in flights_in_frames:
            continue
        flight_list.append({
            'ident': row['ident'],
            'time_appear': pd.to_datetime(row['time_appear']).strftime('%H:%M:%S'),
            'time_disappear': pd.to_datetime(row['time_disappear']).strftime('%H:%M:%S'),
            'alt_min_ft': int(row['alt_min_ft']),
            'alt_max_ft': int(row['alt_max_ft']),
            'label': row['label'] if pd.notna(row['label']) else '',
            'notes': row['notes'] if pd.notna(row['notes']) else '',
            'section': int(row.get('section', 1))
        })
    
    data = {
        'date': DATE_STR,
        'camera': CAMERA_SIDE,
        'image_base_path': base_dir,
        'total_frames': len(frames),
        'frames': frames,
        'flights': flight_list
    }
    
    with open(output_json_path, 'w') as f:
        json.dump(data, f)
    
    print(f"Exported {len(frames)} frames with flight data to {output_json_path}")
    return data

# Export the data
labeling_data = export_labeling_data(
    image_df, df_upsampled, labels_df, BASE_DIR,
    f"./labeling_data_{DATE_STR}_{CAMERA_NAME}.json"
)
print("Data export complete. You can now use the HTML labeler to label the flights.")

Exported 3960 frames with flight data to ./labeling_data_2025-03-13_wisconsin_east.json
Data export complete. You can now use the HTML labeler to label the flights.
